In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# ----------------------- PARAMETRES A REGLER --------------------------
FICHIER_ATTAQUE = "TAnormal.csv"        # fichier de trafic d'attaque (avec les secondes !)
FICHIER_NORMAL  = "traficNormal.csv"    # fichier de trafic normal
SWITCH_MESURE   = "OVS-1_Core"          # point de mesure unique (coeur du reseau)
ATTAQUANTS      = {"2001:4278:19:e9cd::50", "fc00::a00:2"}  # sources d'attaque
FENETRE         = "1s"                   # taille de la fenetre temporelle
LONGUEUR_SEQ    = 10                     # nb de fenetres par sequence (LSTM)
FEATURES = ['n_paq','n_oct','taille_moy','taille_ect',
            'n_dst','n_dport','f_tcp','f_udp','f_icmp']

In [ ]:
# ----------------------------------------------------------------------


def charger(chemin, phase):
    """Charge un CSV du controleur et ne garde que les colonnes utiles."""
    df = pd.read_csv(chemin, dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    utiles = ['timestamp','switch','src_ip','dst_ip',
              'protocole','src_port','dst_port','taille_paquet']
    df = df[[c for c in utiles if c in df.columns]].copy()
    df['phase'] = phase
    return df


def pretraiter(df):
    """Point de mesure unique + typage + familles de protocole + fenetre."""
    # 1) Un seul switch -> chaque paquet compte une seule fois
    df = df[df['switch'] == SWITCH_MESURE].copy()
    # 2) Typage
    df['taille'] = pd.to_numeric(df['taille_paquet'], errors='coerce').fillna(0)
    df['ts'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['ts'])
    df['sec'] = df['ts'].dt.floor(FENETRE)
    # 3) Familles de protocole
    p = df['protocole'].astype(str)
    df['is_tcp']  = p.str.startswith('TCP').astype(int)
    df['is_udp']  = p.str.startswith('UDP').astype(int)
    df['is_icmp'] = (p.str.startswith('PING') | p.str.startswith('ICMP')).astype(int)
    
    return df


def agreger(df):
    """Agrege les paquets en fenetres (phase, source, seconde) + caracteristiques."""
    g = df.groupby(['phase','src_ip','sec'])
    feat = g.agg(
        n_paq      = ('taille','size'),      # paquets par seconde (DEBIT)
        n_oct      = ('taille','sum'),        # octets par seconde
        taille_moy = ('taille','mean'),       # taille moyenne
        taille_ect = ('taille','std'),        # ecart-type de taille
        n_dst      = ('dst_ip','nunique'),    # nb de destinations distinctes
        n_dport    = ('dst_port','nunique'),  # nb de ports destination distincts
        f_tcp      = ('is_tcp','mean'),       # fraction TCP
        f_udp      = ('is_udp','mean'),       # fraction UDP
        f_icmp     = ('is_icmp','mean'),      # fraction ICMP
    ).reset_index()
    feat['taille_ect'] = feat['taille_ect'].fillna(0)
    # Etiquetage : attaque = phase d'attaque ET source attaquante
    feat['label'] = ((feat['phase'] == 'attack') &
                     (feat['src_ip'].isin(ATTAQUANTS))).astype(int)
    print("Nombre total de fenêtres :", len(feat))
    print(feat['label'].value_counts())
    return feat


def construire_sequences(feat, L):
    """Sequences de L fenetres consecutives par source (pour le LSTM)."""
    feat = feat.sort_values(['phase','src_ip','sec']).reset_index(drop=True)
    X, y = [], []
    for (_, _), g in feat.groupby(['phase','src_ip']):
        M = g[FEATURES].values
        lab = g['label'].values
        for i in range(len(g) - L + 1):
            X.append(M[i:i+L])
            y.append(lab[i+L-1])   # label = derniere fenetre de la sequence
    return np.array(X, dtype=float), np.array(y)


def evaluer(nom, y_vrai, y_pred):
    """Affiche les metriques et renvoie un dictionnaire + la matrice."""
    d = {
        'Accuracy':  accuracy_score(y_vrai, y_pred),
        'Precision': precision_score(y_vrai, y_pred),
        'Rappel':    recall_score(y_vrai, y_pred),
        'F1-score':  f1_score(y_vrai, y_pred),
    }
    print(f"\n--- {nom} ---")
    for k, v in d.items():
        print(f"  {k:10s}: {v:.3f}")
    mc = confusion_matrix(y_vrai, y_pred)
    print("  Matrice de confusion:\n", mc)
    return d, mc


def tracer_matrice(mc, titre, fichier):
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    im = ax.imshow(mc, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(mc[i, j]), ha='center', va='center',
                    color='white' if mc[i, j] > mc.max()/2 else 'black',
                    fontsize=13, fontweight='bold')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Normal','Attaque']); ax.set_yticklabels(['Normal','Attaque'])
    ax.set_xlabel('Prediction'); ax.set_ylabel('Verite')
    ax.set_title(titre)
    fig.tight_layout(); fig.savefig(fichier, dpi=150); plt.close(fig)

In [ ]:
# ============================ PIPELINE ================================
print("1) Chargement...")
df = pd.concat([charger(FICHIER_ATTAQUE, 'attack'),
                charger(FICHIER_NORMAL,  'normal')], ignore_index=True)
print("Nombre de paquets :", len(df))

print("2-3) Pretraitement et agregation...")
feat = agreger(pretraiter(df))
print(f"   Fenetres obtenues : {len(feat)}  "
      f"(normal={int((feat.label==0).sum())}, attaque={int((feat.label==1).sum())})")

print("4-5) Construction des sequences...")
X, y = construire_sequences(feat, LONGUEUR_SEQ)
print(f"   Sequences : {X.shape}  (attaque={int(y.sum())}, normal={int((y==0).sum())})")

In [ ]:
# Split identique pour les deux modeles
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                      stratify=y, random_state=42)
print("Entraînement :", len(Xtr), " | Test :", len(Xte))

# Normalisation
sc = StandardScaler().fit(Xtr.reshape(-1, len(FEATURES)))
Xtr_s = sc.transform(Xtr.reshape(-1, len(FEATURES))).reshape(Xtr.shape)
Xte_s = sc.transform(Xte.reshape(-1, len(FEATURES))).reshape(Xte.shape)


In [ ]:
# ---------------------- Random Forest --------------------------------
print("\n6) Entrainement Random Forest...")
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                            random_state=42)
rf.fit(Xtr.reshape(len(Xtr), -1), ytr)      # sequences aplaties
d_rf, mc_rf = evaluer("RANDOM FOREST", yte, rf.predict(Xte.reshape(len(Xte), -1)))

In [ ]:
# ---------------------------- LSTM -----------------------------------
print("\n6) Entrainement LSTM...")
modele = models.Sequential([
    layers.Input((LONGUEUR_SEQ, len(FEATURES))),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
modele.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
poids = {0: 1.0, 1: float((y == 0).sum()) / float(y.sum())}  # gere le desequilibre
hist = modele.fit(Xtr_s, ytr, epochs=25, batch_size=32, verbose=0,
                  class_weight=poids, validation_split=0.1)
y_lstm = (modele.predict(Xte_s, verbose=0).ravel() >= 0.5).astype(int)
d_lstm, mc_lstm = evaluer("LSTM", yte, y_lstm)

In [ ]:
# ---------------------- Figures pour le memoire ----------------------
print("\n7) Generation des figures...")
tracer_matrice(mc_rf,   "Matrice de confusion - Random Forest", "matrice_confusion_rf.png")
tracer_matrice(mc_lstm, "Matrice de confusion - LSTM",          "matrice_confusion_lstm.png")

# Comparaison des metriques (barres)
mets = list(d_rf.keys())
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(mets)); w = 0.35
ax.bar(x - w/2, [d_rf[m] for m in mets],   w, label='Random Forest')
ax.bar(x + w/2, [d_lstm[m] for m in mets], w, label='LSTM')
ax.set_xticks(x); ax.set_xticklabels(mets)
ax.set_ylim(0.8, 1.0); ax.set_ylabel('Score')
ax.set_title('Comparaison Random Forest vs LSTM')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); fig.savefig("comparaison_metriques.png", dpi=150); plt.close(fig)

In [ ]:
# Courbe d'apprentissage du LSTM
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist.history['accuracy'], label='Entrainement')
ax.plot(hist.history['val_accuracy'], label='Validation')
ax.set_xlabel('Epoque'); ax.set_ylabel('Accuracy')
ax.set_title("Courbe d'apprentissage du LSTM"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("courbe_apprentissage_lstm.png", dpi=150); plt.close(fig)

print("\nTermine. Figures enregistrees :")
print("  - matrice_confusion_rf.png")
print("  - matrice_confusion_lstm.png")
print("  - comparaison_metriques.png")
print("  - courbe_apprentissage_lstm.png")

In [ ]:
import joblib

# 1) Sauvegarde du Random Forest
joblib.dump(rf, "modele_rf.pkl")

# 2) Sauvegarde du LSTM
modele.save("modele_lstm.h5")

# 3) Sauvegarde du normaliseur (indispensable !)
joblib.dump(sc, "scaler.pkl")

# 4) Sauvegarde de la liste des caractéristiques (pour garder le bon ordre)
joblib.dump(FEATURES, "features.pkl")

print("Modeles et outils sauvegardes :")
print("  - modele_rf.pkl")
print("  - modele_lstm.h5")
print("  - scaler.pkl")
print("  - features.pkl")

In [ ]:
modele.save_weights("poids_lstm.weights.h5")
from google.colab import files
files.download("poids_lstm.weights.h5")